# 01 – Khám phá dữ liệu (EDA)
Mục tiêu: kiểu dữ liệu, thiếu/trùng, phân phối nhãn, độ dài, từ phổ biến → hình thành giả thuyết.

In [ ]:
import sys
sys.path.insert(0, "..")  # để import được package spam_filter
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.max_colwidth", 110)
pd.set_option("display.width", 200)

from spam_filter.config import DataConfig
from spam_filter.data import load_dataset, split_data

cfg = DataConfig()
df = load_dataset(cfg)
X_train, X_test, y_train, y_test = split_data(df, cfg)

## 1. Chất lượng dữ liệu

In [ ]:
from spam_filter.data import _read_table
raw = _read_table(cfg)
print(raw.dtypes, "\n")
print("Giá trị thiếu:", raw.isna().sum().to_dict())
print("Tin trùng:", raw.duplicated(subset="text").sum())

## 2. Phân phối nhãn (kiểm tra mất cân bằng)

In [ ]:
from spam_filter import eda
display(eda.summarize(df))
df["label"].map({0: "ham", 1: "spam"}).value_counts().plot.bar(color=["#4C72B0", "#C44E52"], rot=0, title="Phân phối nhãn");

## 3. Độ dài tin nhắn theo lớp

In [ ]:
df.assign(chars=df.text.str.len()).groupby("label")["chars"].describe().round(1)

In [ ]:
for label, name, c in ((0, "ham", "#4C72B0"), (1, "spam", "#C44E52")):
    plt.hist(df[df.label == label].text.str.len(), bins=40, alpha=.6, density=True, label=name, color=c)
plt.xlabel("Số ký tự"); plt.legend(); plt.title("Độ dài tin nhắn");

## 4. Dấu hiệu đặc trưng của spam

In [ ]:
feat = pd.DataFrame({
    "url": df.text.str.contains(r"www\.|http|\.com|\.co\.uk", case=False),
    "digits>=5": df.text.str.count(r"\d") >= 5,
    "money": df.text.str.contains(r"[£$€]"),
    "exclaim": df.text.str.contains("!"),
    "ALLCAPS word": df.text.str.contains(r"\b[A-Z]{3,}\b"),
})
feat.groupby(df.label.map({0: "ham", 1: "spam"})).mean().round(3).T

## 5. Từ phổ biến theo lớp

In [ ]:
from collections import Counter
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
for label, name in ((0, "ham"), (1, "spam")):
    words = Counter(w for t in df[df.label == label].text for w in t.lower().split()
                    if w.isalpha() and w not in ENGLISH_STOP_WORDS)
    print(name, words.most_common(15))

## Giả thuyết ban đầu (nhóm điền)
- Lớp có thể mất cân bằng → không nên chỉ nhìn Accuracy.
- So sánh độ dài và các dấu hiệu số/URL/tiền/chữ hoa trước khi chọn tiền xử lý.
- (bổ sung quan sát của nhóm)